# Movie Recommendation and Sentiment Analysis

In [1]:
!pip install tmdbv3api
!pip install bs4
!pip install requests

**1. Import Necessary libraries and download NLTK data**

In [2]:
import os
import time
import random
import requests
import pandas as pd
import numpy as np
import joblib
import re

# For model building and similarity calculation
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# For natural language processing (text cleaning)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk

# For web scraping IMDb reviews
from bs4 import BeautifulSoup

# For displaying DataFrames correctly in Jupyter
from IPython.display import display

# Download NLTK data (only needs to be run once)
print("Downloading NLTK stopwords...")
nltk.download('stopwords')
print("Downloading NLTK WordNet...")
nltk.download('wordnet')
print("NLTK downloads complete.")

# how to hide warning
import warnings
warnings.filterwarnings('ignore')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


NLTK downloads complete.


**2. Load Pre-trained Models and Set API Key**

In [3]:


# Load pre-trained models and data
print("Loading pre-trained models...")
movies = joblib.load('movies.pkl')
transform_vectorizer = joblib.load('transform.pkl')
svc_model = joblib.load('svc_model.pkl') # trained SVC model for sentiment analysis
print("Models loaded successfully.")

# Set your TMDB API key
tmdb_api_key = os.getenv('TMDB_API_KEY')
if not tmdb_api_key:
    raise ValueError("Error: TMDB_API_KEY is not set. Please set the environment variable or replace the placeholder.")
else:
    print("TMDB API key loaded.")

Loading pre-trained models...
Models loaded successfully.
TMDB API key loaded.


**3. Fetch movie posters and IMDb IDs with TMDB API**

In [4]:
def get_imdb_id(tmdb_id):
    """
    Fetches the IMDb ID for a given TMDB movie ID using the TMDB API.

    Args:
        tmdb_id (int): The The Movie Database (TMDB) ID of the movie.

    Returns:
        str or None: The IMDb ID (e.g., 'tt0468569') if found, otherwise None.
    """
    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}?api_key={tmdb_api_key}&language=en-US"
    try:
        response = requests.get(url)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        data = response.json()
        return data.get('imdb_id')
    except requests.exceptions.RequestException as e:
        print(f"Error fetching IMDb ID for TMDB ID {tmdb_id}: {e}")
        return None

def fetch_poster(movie_id):
    """
    Fetches the poster URL for a given TMDB movie ID using the TMDB API.

    Args:
        movie_id (int): The The Movie Database (TMDB) ID of the movie.

    Returns:
        str or None: The URL of the movie poster if available, otherwise None.
    """
    url = f"https://api.themoviedb.org/3/movie/{movie_id}?api_key={tmdb_api_key}&language=en-US"
    try:
        data = requests.get(url).json()
        poster_path = data.get('poster_path')
        if poster_path:
            return f"https://image.tmdb.org/t/p/w500/{poster_path}"
        return None
    except requests.exceptions.RequestException as e:
        print(f"Error fetching poster for TMDB ID {movie_id}: {e}")
        return None

**4. Text Preprocessing and Sentiment Prediction**

In [5]:
# Cell 4: Text Preprocessing and Sentiment Prediction Functions

def remove_stopwords(text):
    """
    Removes common English stopwords from a given text.

    Args:
        text (str): The input text.

    Returns:
        str: The text with stopwords removed.
    """
    stop_words = stopwords.words('english')
    return ' '.join([word for word in text.split() if word not in stop_words])

def perform_lemmatization(text):
    """
    Performs lemmatization on words in a given text.
    Lemmatization reduces words to their base or root form (e.g., "running" -> "run").

    Args:
        text (str): The input text.

    Returns:
        str: The text with words lemmatized.
    """
    lemmatizer = WordNetLemmatizer()
    return ' '.join([lemmatizer.lemmatize(word) for word in text.split()])

def clean_text(text):
    """
    Cleans a given text by performing the following steps:
    1. Removes HTML tags.
    2. Removes non-alphanumeric characters (keeping spaces).
    3. Removes URLs.
    4. Removes digits.
    5. Converts text to lowercase.
    6. Removes extra whitespace.
    7. Removes stopwords.
    8. Performs lemmatization.

    Args:
        text (str): The raw input text.

    Returns:
        str: The cleaned and processed text.
    """
    text = re.sub(r'<[^>]+>', ' ', text) # Remove HTML tags
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text) # Remove non-alphanumeric characters
    text = re.sub(r'http\S+', '', text) # Remove URLs
    text = re.sub(r'\d+', '', text) # Remove digits
    text = text.lower() # Convert to lowercase
    text = re.sub(r'\s+', ' ', text) # Replace multiple spaces with a single space
    text = remove_stopwords(text) # Remove stopwords
    text = perform_lemmatization(text) # Perform lemmatization
    return text

def predict_sentiment(reviews):
    """
    Predicts the sentiment (Positive/Negative) of a list of movie reviews.

    Args:
        reviews (list of str): A list of raw review strings.

    Returns:
        list of str: A list of predicted sentiments ('Positive' or 'Negative')
                     corresponding to each review. Returns an empty list if no reviews.
    """
    if not reviews:
        return []

    # Clean the review texts
    cleaned = [clean_text(r) for r in reviews]

    # Transform the cleaned texts using the pre-trained vectorizer
    transformed = transform_vectorizer.transform(cleaned)

    # Predict labels using the pre-trained SVC model
    labels = svc_model.predict(transformed)

    # Convert numerical labels (0/1) to 'Negative'/'Positive'
    return ['Positive' if label == 1 else 'Negative' for label in labels]

**5. Scrape IMDB movie reviews**

In [6]:
# Cell 5: IMDb Review Scraper Function

def get_reviews_internal(imdb_movie_id):
    """
    Scrapes movie reviews from IMDb given an IMDb movie ID.
    Includes logic to fetch additional reviews via AJAX 'Load More' buttons.

    Args:
        imdb_movie_id (str): The IMDb ID of the movie (e.g., 'tt0468569').

    Returns:
        list of str: A list of scraped review texts. Returns an empty list if no reviews found
                     or if an error occurs.
    """
    base_imdb_url = "https://www.imdb.com"
    reviews_found = []

    # Define headers to mimic a web browser and avoid being blocked
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

    try:
        if not imdb_movie_id:
            print("Error: No IMDb ID provided to get_reviews_internal. Cannot scrape reviews.")
            return []

        # Construct the URL for the movie's review page
        reviews_url = f"{base_imdb_url}/title/{imdb_movie_id}/reviews"
        print(f"Attempting to scrape reviews from: {reviews_url}")

        # Fetch the initial reviews page
        reviews_response = requests.get(reviews_url, headers=headers)
        reviews_response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        reviews_soup = BeautifulSoup(reviews_response.text, 'html.parser')

        # Extract reviews from the initial page
        # Look for div elements with data-testid="review-card-parent"
        review_card_parents = reviews_soup.find_all('div', attrs={'data-testid': 'review-card-parent'})

        if not review_card_parents:
            print(f"Warning: No review parent cards found for IMDb ID '{imdb_movie_id}'. "
                  "This might indicate no reviews available or a change in IMDb's HTML structure.")

        for review_parent_div in review_card_parents:
            # Within each review card, find the div containing the actual review text
            review_text_div = review_parent_div.find('div', class_='ipc-html-content-inner-div')
            if review_text_div:
                text = review_text_div.get_text(strip=True) # Get clean text, remove leading/trailing whitespace
                if text:
                    reviews_found.append(text)

        print(f"Found {len(reviews_found)} reviews on initial page.")

        # Handle "Load More" functionality using AJAX
        pagination_key = None
        # Find the container for the "Load More" button
        load_more_container = reviews_soup.find('div', class_='load-more-reviews')
        if load_more_container:
            # Find the button inside this container, which should have the 'data-key' attribute
            load_more_button = load_more_container.find('button', class_='ipc-btn')
            if load_more_button and 'data-key' in load_more_button.attrs:
                pagination_key = load_more_button['data-key']

        # Loop to fetch reviews from subsequent pages via AJAX
        while pagination_key:
            ajax_url = f"{base_imdb_url}/title/{imdb_movie_id}/reviews/_ajax?ref_=undefined&paginationKey={pagination_key}"
            print(f"Fetching more reviews with pagination key: {pagination_key}")

            # Be polite: wait for a random duration to avoid overwhelming the server
            time.sleep(random.uniform(1.5, 3.5))

            ajax_response = requests.get(ajax_url, headers=headers)
            ajax_response.raise_for_status()

            ajax_soup = BeautifulSoup(ajax_response.text, 'html.parser')

            new_review_card_parents = ajax_soup.find_all('div', attrs={'data-testid': 'review-card-parent'})

            # If no new review cards are found, it means there are no more reviews
            if not new_review_card_parents:
                print("No more review pages found.")
                pagination_key = None
                break

            for review_parent_div in new_review_card_parents:
                review_text_div = review_parent_div.find('div', class_='ipc-html-content-inner-div')
                if review_text_div:
                    text = review_text_div.get_text(strip=True)
                    if text:
                        reviews_found.append(text)

            print(f"Total reviews found so far: {len(reviews_found)}")

            # Find the next pagination key from the newly loaded AJAX content
            next_load_more_container = ajax_soup.find('div', class_='load-more-reviews')
            if next_load_more_container:
                next_load_more_button = next_load_more_container.find('button', class_='ipc-btn')
                if next_load_more_button and 'data-key' in next_load_more_button.attrs:
                    pagination_key = next_load_more_button['data-key']
                else:
                    # No data-key found, implying no more pages
                    pagination_key = None
            else:
                # No 'load-more-reviews' container found, implying no more pages
                pagination_key = None

    except requests.exceptions.RequestException as e:
        print(f"Network error or HTTP issue fetching IMDb page for ID {imdb_movie_id}: {e}")
    except Exception as e:
        print(f"An unexpected error occurred during scraping for ID {imdb_movie_id}: {e}")

    return reviews_found

**6. movie recommendations based on a cosine similarity**

In [7]:
# Cell 6: Movie Recommendation Function

def recommend(movie_name):
    """
    Recommends 5 movies similar to the given movie name based on content similarity.

    Args:
        movie_name (str): The title of the movie for which to find recommendations.

    Returns:
        list of tuple: A list of (movie_title, poster_url, tmdb_movie_id) tuples for the
                       top 5 recommended movies. Returns an empty list if the movie
                       is not found in the database.
    """
    if movie_name not in movies['movie_title'].values:
        print(f"Error: Movie '{movie_name}' not found in the recommendation database.")
        return []

    # Get the index of the selected movie
    index = movies[movies['movie_title'] == movie_name].index[0]

    # Re-calculate CountVectorizer and cosine similarity.
    # Note: For efficiency in a larger application, 'vectors' and 'similarity'
    # could be pre-calculated and loaded, rather than re-calculating on each call.
    # However, for this example, re-calculation ensures fresh vectors if 'movies' changes.
    cv = CountVectorizer()
    vectors = cv.fit_transform(movies['comb']) # 'comb' likely represents combined features like genres, cast, crew
    similarity = cosine_similarity(vectors)

    # Get similarity scores for the selected movie with all other movies
    distances = sorted(list(enumerate(similarity[index])), reverse=True, key=lambda x: x[1])

    recommendations = []
    # Iterate through the top 5 most similar movies (excluding the movie itself, hence [1:6])
    for i in distances[1:6]:
        tmdb_movie_id = movies.iloc[i[0]].movie_id # Get TMDB ID from your movie dataframe
        poster_url = fetch_poster(tmdb_movie_id) # Fetch poster using the TMDB ID
        title = movies.iloc[i[0]].movie_title
        recommendations.append((title, poster_url, tmdb_movie_id))

    return recommendations

**7. getting recommendations, fetching reviews (using TMDB for IMDb ID), and performing sentiment analysis**

In [8]:
# Cell 7: Main Execution Function

def movie_recommendation_and_sentiments(selected_movie):
    """
    Provides movie recommendations and performs sentiment analysis on reviews
    for a given movie.

    Args:
        selected_movie (str): The title of the movie to process.
    """
    print(f"--- Processing for: {selected_movie} ---")

    # 1. Get TMDB ID for the selected movie from your local 'movies' dataframe
    selected_movie_df = movies[movies['movie_title'] == selected_movie]
    if selected_movie_df.empty:
        print(f"Error: Movie '{selected_movie}' not found in the local movie database. "
              "Cannot provide recommendations or sentiment analysis.")
        return

    selected_tmdb_id = selected_movie_df.iloc[0].movie_id
    print(f"Found TMDB ID: {selected_tmdb_id} for '{selected_movie}'")

    # 2. Use the TMDB ID to get the IMDb ID (via TMDB API)
    imdb_id_for_reviews = get_imdb_id(selected_tmdb_id)
    if imdb_id_for_reviews:
        print(f"Retrieved IMDb ID: {imdb_id_for_reviews} from TMDB.")
    else:
        print(f"Warning: Could not get IMDb ID from TMDB for movie '{selected_movie}'. "
              "Reviews cannot be scraped. Skipping sentiment analysis for this movie.")

    reviews = []
    if imdb_id_for_reviews:
        # 3. Pass the obtained IMDb ID to get_reviews_internal to scrape reviews
        reviews = get_reviews_internal(imdb_id_for_reviews)

    # Display Recommendations
    print(f"\n🎬 Recommendations for: {selected_movie}")
    recommended_movies = recommend(selected_movie)
    if recommended_movies:
        for title, poster, _ in recommended_movies: # Underscore to ignore the TMDB ID returned from recommend
            if poster:
                print(f"{title}\nPoster: {poster}\n")
            else:
                print(f"{title}\nPoster: Not available\n")
    else:
        print("No recommendations found.")

    # Display Sentiment Analysis
    if reviews:
        sentiments = predict_sentiment(reviews)
        df_sentiments = pd.DataFrame({'Review': reviews, 'Sentiment': sentiments})
        print("\nSentiment Analysis of Reviews:")

        # Configure Pandas display options for better visibility in Jupyter
        pd.set_option('display.max_rows', None)        # Display all rows
        pd.set_option('display.max_colwidth', None)   # Display full content of columns

        # Use display() for proper, rich HTML table rendering in Jupyter Notebook
        display(df_sentiments)
    else:
        print("❌ No reviews found for sentiment analysis.")

    print(f"--- End of processing for: {selected_movie} ---")

**8. Test recommendations and sentiment of reviews for some movies**

In [9]:
Movie = "The Dark Knight"
movie_recommendation_and_sentiments(Movie)

--- Processing for: The Dark Knight ---
Found TMDB ID: 155.0 for 'The Dark Knight'
Retrieved IMDb ID: tt0468569 from TMDB.
Attempting to scrape reviews from: https://www.imdb.com/title/tt0468569/reviews
Found 18 reviews on initial page.

🎬 Recommendations for: The Dark Knight
The Dark Knight Rises
Poster: https://image.tmdb.org/t/p/w500//hr0L2aueqlP2BYUblTTjmtn0hw4.jpg

Batman Begins
Poster: https://image.tmdb.org/t/p/w500//4MpN4kIEqUjW8OPtOQJXlTdHiJV.jpg

The Prestige
Poster: https://image.tmdb.org/t/p/w500//bdN3gXuIZYaJP7ftKK2sU0nPtEA.jpg

London Has Fallen
Poster: https://image.tmdb.org/t/p/w500//iEbLkYzyiUdOKNK4WNBFyGH7r2Y.jpg

American Psycho
Poster: https://image.tmdb.org/t/p/w500//9uGHEgsiUXjCNq8wdq4r49YL8A1.jpg


Sentiment Analysis of Reviews:


,Review,Sentiment
0,Best movie ever. Heath ledger's work is phenomenal no words......,Positive
1,"This movie is a work of art. The finest sequel ever made. I don't think we will see another movie like this for a long time. Heath Ledger's Joker is the best movie charachter I have ever seen by far. Avengers Endgame is great, but The Dark Knight is much better. The best Batman ever! The best Joker ever! The best DC movie ever! The best superhero movie ever! Ando for me, the best movie ever!",Positive
2,"It is just what you want for the best movie. Great story great acting, thrilling twist.\nJust watched Joker in 2019, I just has to come back and give dark knight a 10. And thanks to Heath Ledger for the exceptional performs.",Positive
3,"Confidently directed, dark, brooding, and packed with impressive action sequences and a complex story, The Dark Knight includes a career-defining turn from Heath Ledger as well as other Oscar worthy performances, TDK remains not only the best Batman movie, but comic book movie ever created.",Positive
4,"We've been subjected to enormous amounts of hype and marketing for the Dark Knight. We've seen Joker scavenger hunts and one of the largest viral campaigns in advertising history and it culminates with the actual release of the movie.Everything that's been said is pretty much spot on. This is the first time I can remember where a summer blockbuster film far surpasses the hype.For as much action as there is in this movie, it's the acting that makes it a great piece of work. Between all the punches, explosions and stunt-work is some great dialog work. All the actors have their moments.Bale's Batman is the definitive Batman because we see everything in this character finally on film. Martial arts skills, cunning, great tactical thinking, forensic application, technological genius to advance or improve Luscious Fox's inventions/technological breakthroughs, intimidating personality, and even a little swashbuckling.As for Heath, yes he gets credit for his performance as the Joker. But you have to also recognize Jonathan and Chris Nolan for the writing and treatment of the character. It's not just the fact that Ledger makes the Joker so menacing, but the Nolans have given the character this great manifesto that drives its actions. The Joker's stance on chaos, order, anarchy, the morality of the average modern human being make the character so interesting psychologically. The Nolans drafted a complex character and only a perfect performance could've pulled something like this off. That's how difficult of a role this was, and that's why Ledger's performance is so great.This isn't an action movie. It's a film that explores literary themes of the hero and villain, as well as order and anarchy. Yes, listen to the dialog because it's all in there.",Positive
5,"I couldn't believe ""The Dark knight"" could live up to the hype. That's perhaps the biggest surprise. The secret, I believe, is a stunning, mature, intelligent script. That makes it the best superhero movie ever made. As if that wasn't enough, Heath Ledger. He, the newest of the tragic modern icons present us with a preview of something we'll never see. A fearless, extraordinary actor capable to fill up with humanity even the most grotesque of villains. His performance is a master class. Fortunately, Christian Bale's Batman is almost a supporting character. Bale is good but there is something around his mouth that stops him from being great. ""The Dark Knight"" is visually stunning, powerful and moving. What else could anyone want.",Positive
6,"Dark, yes, complex, ambitious. Christopher Nolan and his co-writer Jonathan Nolan deserve a standing ovation. I don't usually go for loud movies filled with mindless gore and violence. ""The Dark Knight"" is certainly loud and violent but it's not mindless. It has depth and soul. Even the Joker, in an extraordinary creation by Heath Ledger, is deeply human. The natural petulance of Christian Bale makes his ego and

--- End of processing for: The Dark Knight ---


In [10]:
Movie = "Barbie"
movie_recommendation_and_sentiments(Movie)

--- Processing for: Barbie ---
Found TMDB ID: 346698.0 for 'Barbie'
Retrieved IMDb ID: tt1517268 from TMDB.
Attempting to scrape reviews from: https://www.imdb.com/title/tt1517268/reviews
Found 15 reviews on initial page.

🎬 Recommendations for: Barbie
The Suicide Squad
Poster: https://image.tmdb.org/t/p/w500//q61qEyssk2ku3okWICKArlAdhBn.jpg

The Sisterhood of the Traveling Pants 2
Poster: https://image.tmdb.org/t/p/w500//bAdkVOD91jCHos4163qyVu1TFua.jpg

Babylon
Poster: https://image.tmdb.org/t/p/w500//wjOHjWCUE0YzDiEzKv8AfqHj3ir.jpg

Lady Bird
Poster: https://image.tmdb.org/t/p/w500//gl66K7zRdtNYGrxyS2YDUP5ASZd.jpg

I, Tonya
Poster: https://image.tmdb.org/t/p/w500//6gNXwSHxaksR1PjVZRqNapmkgj3.jpg


Sentiment Analysis of Reviews:


,Review,Sentiment
0,"Margot does the best with what she's given, but this film was very disappointing to me. It was marketed as a fun, quirky satire with homages to other movies. It started that way, but ended with over-dramatized speeches and an ending that clearly tried to make the audience feel something, but left everyone just feeling confused. And before you say I'm a crotchety old man, I'm a woman in my 20s, so I'm pretty sure I'm this movie's target audience. The saddest part is there were parents with their kids in the theater that were victims of the poor marketing, because this is not a kid's movie. Overall, the humor was fun on occasion and the film is beautiful to look at, but the whole concept falls apart in the second half of the film and becomes a pity party for the ""strong"" woman.",Negative
1,"I do not usually write reviews, but this is beyond description. Last time I wrote a review it was for the absolutely horrible wrinkle in time. This is almost as bad. At least Gosling plays his part well, but the story is abysmal. If you can find anything else to do like watch paint dry, then yes please do. You will save your brain cells the agony of a painful slow ..... nearly two hours long this was mind-numbing storytelling at its mediocre worst. How many ways can I say do not go see it if you have a choice. See anything else but this exercise in horrendous cliches and exaggerated self-importance. If I could give it negative 10 stars I would.",Negative
2,"Seeing a lot of reviews saying that this movie is preachy, the men are portrayed as idiots, etc. It's a SATIRE. It is meant to be over the top. And yes, it has a lot to say. But the fact that some people in the audience (and I'd wager some of the negative reviews are from people who didn't bother to watch the movie) don't want to hear it, doesn't make it any less valid.In Barbieland, Ken is just an accessory to Barbie and is portrayed as nothing more than an object - something women in the real world deal with all the time. The fact that so many men are reacting so negatively to this movie tells me they don't like being treated the way that women are treated every day. And that is the point of the movie - yet it has much more to say than that. Ultimately, the message in Barbie isn't that ""women rule and men drool"" - it's that patriarchy ultimately harms both men and women. And that's true. So people who say this movie is anti-men are missing the entire point. Patriarchy may give men privilege but it ultimately hurts them, too - pressuring them to conform to a rigid idea of manhood and silencing their self-expression.I am living for the fact that this movie is unapologetic about showing the absurdity of it all. I would have given it a 10 if it weren't for the cringe worthy Ken dance number. It was the only weak part of the movie. But otherwise, Bravo!",Negative
3,"The quality, the humor, and the writing of the movie is fun for a while. It's quirky and it's unique. When they get into the weeds and try to explore deeper themes, the movie is a miss. The middle expositional phase of the movie, I must say, is a bore.The movie was so close to being great, but it was incapable of delivering a message without having a character literally give a speech worth of dialogue.What could have been a great satire and what began as a cleverly funny movie turned into a preachy and moralizing movie in the end. But I guess the charm of the first half of the movie made up for the shortcomings.",Positive
4,"I walked out of the theatre thinking, ""Yeah, I had a good time in that movie"". But as the day went on I kind had that ""Ok, that kebab probably wasn't a good idea.""The film is stunning in its production design and creativity, and Margot Robbie is perfectly cast as the Stereotypical Barbie. For the most part, the songs are catchy and performances are solid for the type of characters being portraying.When we strip away the outside world and all the messaging, Barbie and Ken have a few

--- End of processing for: Barbie ---


In [11]:
import os
import time
import random
import requests
import pandas as pd
import numpy as np
import joblib
import re

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk

from bs4 import BeautifulSoup

nltk.download('stopwords')
nltk.download('wordnet')


movies = joblib.load('movies.pkl')
transform_vectorizer = joblib.load('transform.pkl')
svc_model = joblib.load('svc_model.pkl')

tmdb_api_key = '613b9e66c1e1b3fee798437e9803e1b5'
if not tmdb_api_key:
    raise ValueError("Please set the TMDB_API_KEY environment variable.")

# This function is correct as is, it fetches IMDb ID using TMDB movie ID
def get_imdb_id(tmdb_id):
    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}?api_key={tmdb_api_key}&language=en-US"
    response = requests.get(url)
    data = response.json()
    return data.get('imdb_id')

def fetch_poster(movie_id):
    url = f"https://api.themoviedb.org/3/movie/{movie_id}?api_key={tmdb_api_key}&language=en-US"
    data = requests.get(url).json()
    poster_path = data.get('poster_path')
    if poster_path:
        return f"https://image.tmdb.org/t/p/w500/{poster_path}"
    return None

def remove_stopwords(text):
    stop_words = stopwords.words('english')
    return ' '.join([word for word in text.split() if word not in stop_words])

def perform_lemmatization(text):
    lemmatizer = WordNetLemmatizer()
    return ' '.join([lemmatizer.lemmatize(word) for word in text.split()])

def clean_text(text):
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = remove_stopwords(text)
    text = perform_lemmatization(text)
    return text

def predict_sentiment(reviews):
    if not reviews:
        return []
    cleaned = [clean_text(r) for r in reviews]
    transformed = transform_vectorizer.transform(cleaned)
    labels = svc_model.predict(transformed)
    return ['Positive' if label == 1 else 'Negative' for label in labels]

def get_reviews_internal(imdb_movie_id): # This function now expects an IMDb ID
    base_imdb_url = "https://www.imdb.com"
    reviews_found = []

    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

    try:
        # Step 1 (Skipped): We already have the IMDb ID, no need to search IMDb for it.
        # This function now assumes a valid IMDb ID is passed to it.
        if not imdb_movie_id:
            print("No IMDb ID provided to get_reviews_internal.")
            return []

        # Construct the URL for the movie's review page using the provided IMDb ID
        reviews_url = f"{base_imdb_url}/title/{imdb_movie_id}/reviews"

        # Step 2: Fetch the reviews page
        reviews_response = requests.get(reviews_url, headers=headers)
        reviews_response.raise_for_status()
        reviews_soup = BeautifulSoup(reviews_response.text, 'html.parser')

        # Step 3: Extract reviews using the provided HTML structure
        review_card_parents = reviews_soup.find_all('div', attrs={'data-testid': 'review-card-parent'})

        if not review_card_parents:
            print(f"No review parent cards found for IMDb ID '{imdb_movie_id}' at {reviews_url}. This might indicate no reviews or a change in the review page HTML structure.")

        for review_parent_div in review_card_parents:
            review_text_div = review_parent_div.find('div', class_='ipc-html-content-inner-div')
            if review_text_div:
                text = review_text_div.get_text(strip=True)
                if text:
                    reviews_found.append(text)

        # Handle "Load More" (AJAX)
        pagination_key = None
        load_more_container = reviews_soup.find('div', class_='load-more-reviews')
        if load_more_container:
            load_more_button = load_more_container.find('button', class_='ipc-btn')
            if load_more_button and 'data-key' in load_more_button.attrs:
                pagination_key = load_more_button['data-key']

        while pagination_key:
            ajax_url = f"{base_imdb_url}/title/{imdb_movie_id}/reviews/_ajax?ref_=undefined&paginationKey={pagination_key}"

            time.sleep(random.uniform(1.5, 3.5))

            ajax_response = requests.get(ajax_url, headers=headers)
            ajax_response.raise_for_status()

            ajax_soup = BeautifulSoup(ajax_response.text, 'html.parser')

            new_review_card_parents = ajax_soup.find_all('div', attrs={'data-testid': 'review-card-parent'})

            if not new_review_card_parents:
                pagination_key = None
                break

            for review_parent_div in new_review_card_parents:
                review_text_div = review_parent_div.find('div', class_='ipc-html-content-inner-div')
                if review_text_div:
                    text = review_text_div.get_text(strip=True)
                    if text:
                        reviews_found.append(text)

            next_load_more_container = ajax_soup.find('div', class_='load-more-reviews')
            if next_load_more_container:
                next_load_more_button = next_load_more_container.find('button', class_='ipc-btn')
                if next_load_more_button and 'data-key' in next_load_more_button.attrs:
                    pagination_key = next_load_more_button['data-key']
                else:
                    pagination_key = None
            else:
                pagination_key = None

    except requests.exceptions.RequestException as e:
        print(f"Network error or HTTP issue fetching IMDb page for ID {imdb_movie_id}: {e}")
    except Exception as e:
        print(f"An unexpected error occurred during scraping for ID {imdb_movie_id}: {e}")

    return reviews_found


def recommend(movie_name):
    if movie_name not in movies['movie_title'].values:
        print(f"Movie '{movie_name}' not found in the database.")
        return []

    index = movies[movies['movie_title'] == movie_name].index[0]

    cv = CountVectorizer()
    vectors = cv.fit_transform(movies['comb'])
    similarity = cosine_similarity(vectors)

    distances = sorted(list(enumerate(similarity[index])), reverse=True, key=lambda x: x[1])

    recommendations = []
    for i in distances[1:6]:
        # --- IMPORTANT CHANGE HERE: Store TMDB ID to use get_imdb_id later ---
        tmdb_movie_id = movies.iloc[i[0]].movie_id # Assuming 'movie_id' in movies.pkl is TMDB ID
        poster_url = fetch_poster(tmdb_movie_id) # Use TMDB ID for poster
        title = movies.iloc[i[0]].movie_title
        recommendations.append((title, poster_url, tmdb_movie_id)) # Add tmdb_movie_id to recommendations
    return recommendations

def movie_recommendation_and_sentiments(selected_movie):
    # 1. Get TMDB ID for the selected movie from your local 'movies' dataframe
    # This assumes 'movies.pkl' contains both movie_title and TMDB movie_id
    selected_movie_df = movies[movies['movie_title'] == selected_movie]
    if selected_movie_df.empty:
        print(f"Movie '{selected_movie}' not found in the local database for sentiment analysis.")
        return

    # Get the TMDB ID for the selected_movie
    selected_tmdb_id = selected_movie_df.iloc[0].movie_id

    # 2. Use the TMDB ID to get the IMDb ID
    imdb_id_for_reviews = get_imdb_id(selected_tmdb_id)

    reviews = []
    if imdb_id_for_reviews:
        # 3. Pass the obtained IMDb ID to get_reviews_internal
        reviews = get_reviews_internal(imdb_id_for_reviews)
    else:
        print(f"Could not get IMDb ID from TMDB for movie '{selected_movie}'. Skipping review scraping.")


    print(f"\n🎬 Recommendations for: {selected_movie}")
    # The 'recommend' function now returns TMDB IDs, which is good.
    # We still want to use the TMDB ID for fetching posters for recommendations.
    for title, poster, _ in recommend(selected_movie): # _ to ignore the TMDB ID from recommend for display
        if poster:
            print(f"{title}\nPoster: {poster}\n")
        else:
            print(f"{title}\nPoster: Not available\n")

    if reviews:
        sentiments = predict_sentiment(reviews)
        df_sentiments = pd.DataFrame({'Review': reviews, 'Sentiment': sentiments})
        print("\nSentiment Analysis of Reviews:")
        pd.set_option('display.max_rows', None)
        pd.set_option('display.max_colwidth', None)
        display(df_sentiments)

    else:
        print("❌ No reviews found for sentiment analysis.")



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [12]:
# Call the main function
movie_recommendation_and_sentiments('Pirates of the Caribbean: The Curse of the Black Pearl')


🎬 Recommendations for: Pirates of the Caribbean: The Curse of the Black Pearl
Pirates of the Caribbean: At World's End
Poster: https://image.tmdb.org/t/p/w500//jGWpG4YhpQwVmjyHEGkxEkeRf0S.jpg

Pirates of the Caribbean: Dead Man's Chest
Poster: https://image.tmdb.org/t/p/w500//lAhcKRt0ggTFkeFL95jrGQYaRXs.jpg

Pirates of the Caribbean: Dead Men Tell No Tales
Poster: https://image.tmdb.org/t/p/w500//qwoGfcg6YUS55nUweKGujHE54Wy.jpg

The Lone Ranger
Poster: https://image.tmdb.org/t/p/w500//xRmsqvHnaWmrazJl9bTBqs4LAjp.jpg

Pirates of the Caribbean: On Stranger Tides
Poster: https://image.tmdb.org/t/p/w500//keGfSvCmYj7CvdRx36OdVrAEibE.jpg


Sentiment Analysis of Reviews:


Review  \
0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [13]:
movie_recommendation_and_sentiments('Barbie')


🎬 Recommendations for: Barbie
The Suicide Squad
Poster: https://image.tmdb.org/t/p/w500//q61qEyssk2ku3okWICKArlAdhBn.jpg

The Sisterhood of the Traveling Pants 2
Poster: https://image.tmdb.org/t/p/w500//bAdkVOD91jCHos4163qyVu1TFua.jpg

Babylon
Poster: https://image.tmdb.org/t/p/w500//wjOHjWCUE0YzDiEzKv8AfqHj3ir.jpg

Lady Bird
Poster: https://image.tmdb.org/t/p/w500//gl66K7zRdtNYGrxyS2YDUP5ASZd.jpg

I, Tonya
Poster: https://image.tmdb.org/t/p/w500//6gNXwSHxaksR1PjVZRqNapmkgj3.jpg


Sentiment Analysis of Reviews:


,Review,Sentiment
0,"Margot does the best with what she's given, but this film was very disappointing to me. It was marketed as a fun, quirky satire with homages to other movies. It started that way, but ended with over-dramatized speeches and an ending that clearly tried to make the audience feel something, but left everyone just feeling confused. And before you say I'm a crotchety old man, I'm a woman in my 20s, so I'm pretty sure I'm this movie's target audience. The saddest part is there were parents with their kids in the theater that were victims of the poor marketing, because this is not a kid's movie. Overall, the humor was fun on occasion and the film is beautiful to look at, but the whole concept falls apart in the second half of the film and becomes a pity party for the ""strong"" woman.",Negative
1,"I do not usually write reviews, but this is beyond description. Last time I wrote a review it was for the absolutely horrible wrinkle in time. This is almost as bad. At least Gosling plays his part well, but the story is abysmal. If you can find anything else to do like watch paint dry, then yes please do. You will save your brain cells the agony of a painful slow ..... nearly two hours long this was mind-numbing storytelling at its mediocre worst. How many ways can I say do not go see it if you have a choice. See anything else but this exercise in horrendous cliches and exaggerated self-importance. If I could give it negative 10 stars I would.",Negative
2,"Seeing a lot of reviews saying that this movie is preachy, the men are portrayed as idiots, etc. It's a SATIRE. It is meant to be over the top. And yes, it has a lot to say. But the fact that some people in the audience (and I'd wager some of the negative reviews are from people who didn't bother to watch the movie) don't want to hear it, doesn't make it any less valid.In Barbieland, Ken is just an accessory to Barbie and is portrayed as nothing more than an object - something women in the real world deal with all the time. The fact that so many men are reacting so negatively to this movie tells me they don't like being treated the way that women are treated every day. And that is the point of the movie - yet it has much more to say than that. Ultimately, the message in Barbie isn't that ""women rule and men drool"" - it's that patriarchy ultimately harms both men and women. And that's true. So people who say this movie is anti-men are missing the entire point. Patriarchy may give men privilege but it ultimately hurts them, too - pressuring them to conform to a rigid idea of manhood and silencing their self-expression.I am living for the fact that this movie is unapologetic about showing the absurdity of it all. I would have given it a 10 if it weren't for the cringe worthy Ken dance number. It was the only weak part of the movie. But otherwise, Bravo!",Negative
3,"The quality, the humor, and the writing of the movie is fun for a while. It's quirky and it's unique. When they get into the weeds and try to explore deeper themes, the movie is a miss. The middle expositional phase of the movie, I must say, is a bore.The movie was so close to being great, but it was incapable of delivering a message without having a character literally give a speech worth of dialogue.What could have been a great satire and what began as a cleverly funny movie turned into a preachy and moralizing movie in the end. But I guess the charm of the first half of the movie made up for the shortcomings.",Positive
4,"I walked out of the theatre thinking, ""Yeah, I had a good time in that movie"". But as the day went on I kind had that ""Ok, that kebab probably wasn't a good idea.""The film is stunning in its production design and creativity, and Margot Robbie is perfectly cast as the Stereotypical Barbie. For the most part, the songs are catchy and performances are solid for the type of characters being portraying.When we strip away the outside world and all the messaging, Barbie and Ken have a few